In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Random Forests and Hyperparameter Tuning
We'll check now how to use and tune RandomForest

## Build a Model Pipeline
Let's load the clean Airbnb dataset in again 
We created it in a previous notebook, it should exists in `/home/jovyan/work/datasets/output/airbnb/clean_data`

In [2]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

file_path = f"/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x+"Index" for x in categorical_cols]

string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")

numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "price"))]

assembler_inputs = index_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

rf = RandomForestRegressor(labelCol="price", maxBins=250)
stages = [string_indexer, vec_assembler, rf]
pipeline = Pipeline(stages=stages)

## ParamGrid

In [3]:
print(rf.explainParams())

bootstrap: Whether bootstrap samples are used when building trees. (default: True)
cacheNodeIds: If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval. (default: False)
checkpointInterval: set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext. (default: 10)
featureSubsetStrategy: The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'onethird' for regression), 'all' (use all features), 'onethird' (use 1/3 of the featur

There are a lot of hyperparameters we could tune, and it would take a long time to manually configure.
We can define a grid of hyperparameters to test:
  - **`maxDepth`**: max depth of each decision tree between **`2 and 5`**)
  - **`numTrees`**: number of decision trees to train between **`5 and 10`**)
**`addGrid()`** requires the name of the parameter as first input (e.g. **`rf.maxDepth`**), and a list of the possible values (e.g. **`[2, 5]`**).

In [5]:
from pyspark.ml.tuning import ParamGridBuilder

param_grid = (ParamGridBuilder()
              .addGrid(rf.maxDepth, [2,5])
              .addGrid(rf.numTrees, [5,10])
              .build())

We pass in the **`estimator`** (pipeline), **`evaluator`**, and **`estimatorParamMaps`** to <a href="https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html?highlight=crossvalidator#pyspark.ml.tuning.CrossValidator" target="_blank">CrossValidator</a> so that it knows:
- Which model to use
- How to evaluate the model
- What hyperparameters to set for the model
We can also set the number of folds we want to split our data into (3), as well as setting a seed so we all have the same split in the data.

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator

evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")

cv = CrossValidator(estimator=pipeline,
                    evaluator=evaluator,
                    estimatorParamMaps=param_grid, 
                    numFolds=3, seed=42)

This param grid will cause to evaluate the model with the following hyperparameters combination:
* maxDepth: 2 numTrees: 5
* maxDepth: 2 numTrees: 10
* maxDepth: 5 numTrees: 5
* maxDepth: 5 numTrees: 10
So four combinations

In [7]:
cv_model = cv.fit(train_df)

Since we have things like StringIndexer (an estimator) in the pipeline, it will be recalculated entirely if the pipeline is put in the cross validator. And that step doesn't matter for the cross validation step
* We can put the only piece that changes (Regressor itself) in the cross validation

In [8]:
cv = CrossValidator(estimator=rf,
                    evaluator=evaluator,
                    estimatorParamMaps=param_grid, 
                    numFolds=3, seed=42)


pipeline = Pipeline(stages=[string_indexer, vec_assembler, cv])

pipeline_model = pipeline.fit(train_df)

We can look at the model with the best hyperparameter configuration by checking it's metrics

In [9]:
list(zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics))

[({Param(parent='RandomForestRegressor_d7775ee6f4f2', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
   Param(parent='RandomForestRegressor_d7775ee6f4f2', name='numTrees', doc='Number of trees to train (>= 1).'): 5},
  45.94037745185954),
 ({Param(parent='RandomForestRegressor_d7775ee6f4f2', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2,
   Param(parent='RandomForestRegressor_d7775ee6f4f2', name='numTrees', doc='Number of trees to train (>= 1).'): 10},
  45.10953033768347),
 ({Param(parent='RandomForestRegressor_d7775ee6f4f2', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 5,
   Param(parent='RandomForestRegressor_d7775ee6f4f2', name='

In [10]:
pred_df = pipeline_model.transform(test_df)

rmse = evaluator.evaluate(pred_df)
r2 = evaluator.setMetricName("r2").evaluate(pred_df)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")

RMSE is 39.85587758352462
R2 is 0.5236665074109967


In [29]:
cv_model = pipeline_model.stages[-1]
best_model = cv_model.bestModel

be